# Day 20 — Analytics SQL: CTEs, Window Functions, JSONB & pandas read_sql (13 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: answer the questions a content team really asks, in SQL that a teammate can read. CTEs split a long query into named steps, window functions give "latest version per entry", rankings and week-over-week change without losing rows, JSONB lets you search inside the flexible CMS fields, and pandas turns a query result into the report people open in Excel.

- Deliverable: day20/analytics.sql with eight queries (A1 to A8) checked by 8 given tests, day20/report.py (a weekly pandas report, 14 given tests), and day20/dsa.py.

- Where this sits: Week 3, Phase 1 (Foundation), day 6 of 7. Days 15 to 19 built and loaded the tables; today you read them like an analyst. Tomorrow (Day 21) is the weekly review. In Week 4 these queries become FastAPI endpoints, and in Phase 2 an LLM agent calls functions like fetch_daily() as tools to answer "how did the Diwali campaign do?".

Time split (5h): the Week 3 split, with the catch-up hour kept.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 5, running every example against setup.sql |
| Hands-on | 2h | analytics.sql (A1 to A8) and report.py until both test files pass |
| Catch-up | 1h | Day 19: upsert_loader.py (21 tests). If it is done, Day 13 re-read as the lane planned |
| DSA | 0.5h | latest_by_key and max_window_sum in dsa.py |

Open items first: GitHub main still ends at Day 4's starters, and Days 5 to 19 reached you as patches and zips because the scheduled push is blocked. Apply them in day order with git am, then git push, before you start. Today's folder ships its own data and does not depend on them.

Before you start: Postgres running and the fde database from Day 15. Then, inside day20/:

```bash
# One-time setup for Day 20, run inside day20/ (macOS / Linux shell).
python3 -m venv .venv          # a fresh virtual environment (Day 6)
source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install -r requirements.txt   # psycopg 3, SQLAlchemy 2, pandas, pytest
export DATABASE_URL=postgresql://localhost/fde   # same database as Day 15
psql "$DATABASE_URL" -f setup.sql        # lesson tables for the concepts
psql "$DATABASE_URL" -f task_data.sql    # Acme's tables for the task
```

setup.sql creates four small lesson tables: lesson_versions (every save of 4 entries), lesson_views (daily views for 5 days), lesson_entries (CMS fields as JSONB) and lesson_pages (a page tree). They are small on purpose, so you can check every output by hand. Every output below came from running the code on PostgreSQL 16 with psycopg 3.3, SQLAlchemy 2.1 and pandas 3.0. Re-run setup.sql whenever the lesson data looks wrong.

## Recap (2 minutes, from memory)

Three questions on Day 19. Answer out loud first; the answers are on the last page.

- A migration adds CHECK (word_count >= 0) to a table where one row has word_count NULL. Does the ALTER TABLE succeed? Why?

- You have an index on (entry_uid, viewed_on). Which can use it: a query filtering only on entry_uid, one filtering only on viewed_on, or one filtering on both?

- Why does a sync upsert end with WHERE ... IS DISTINCT FROM EXCLUDED ... instead of <>, and what would go wrong without any WHERE at all?

## Concept 1: CTEs, queries written as named steps

### Why it matters for an FDE

Real customer questions are rarely one SELECT. "Which campaigns got more views than average, and what is their latest title?" needs a total per entry, an average of those totals, the latest version of each entry, and then a filter. Written as nested subqueries, that is a block you read from the inside out and nobody wants to review.

A CTE (common table expression) lets you write the same query as a list of named steps, top to bottom. This matters for an FDE because your SQL is read by other people: the customer's data team, the teammate who takes over the engagement, and the interviewer watching you share your screen. Readable SQL gets reviewed, trusted and reused.

### How it works

A CTE starts with WITH name AS ( query ). After that, the final SELECT (and any later CTE) can use name like a table. You can chain several, separated by commas. Think of them as the const lines at the top of a React component: each one computes a piece with a clear name, and the return at the bottom puts them together.

```sql
-- Concept 1: the same question as nested subqueries, then as CTEs.
-- Question: which entries had more total views than the average entry?
-- 1. Nested subqueries: you have to read from the inside out.
SELECT uid, total
FROM (SELECT uid, sum(views) AS total FROM lesson_views GROUP BY uid) t
WHERE total > (SELECT avg(total) FROM
              (SELECT sum(views) AS total FROM lesson_views GROUP BY uid) a)
ORDER BY total DESC;
-- ->   uid   | total
-- ->  blt101 |   935
-- 2. With CTEs: read top to bottom, one named step at a time.
WITH totals AS (                    -- step 1: views per entry
    SELECT uid, sum(views) AS total
    FROM lesson_views
    GROUP BY uid
),
average AS (                        -- step 2: one number, the average
    SELECT avg(total) AS avg_total FROM totals   -- reuses step 1
)
SELECT t.uid, t.total, round(a.avg_total, 1) AS avg_total
FROM totals t
CROSS JOIN average a    -- average has 1 row, so this adds 1 column
WHERE t.total > a.avg_total
ORDER BY t.total DESC;
-- ->   uid   | total | avg_total
-- ->  blt101 |   935 |     409.3
-- Same answer. Notice the subquery version had to compute the totals
-- twice; the CTE version names them once and uses them twice.
```

A CTE lives only inside its own statement. It is not saved anywhere, which is different from a table or a view.

```sql
-- Concept 1: a CTE disappears when its statement ends.
WITH totals AS (
    SELECT uid, sum(views) AS total FROM lesson_views GROUP BY uid
)
SELECT count(*) FROM totals;
-- -> 4
SELECT count(*) FROM totals;          -- a NEW statement: totals is gone
-- -> ERROR:  relation "totals" does not exist
-- FDE TIP: if many queries need the same step, make it a VIEW
-- (CREATE VIEW latest_versions AS ...). A view is a saved query.
```

CTEs have one more power: WITH RECURSIVE can walk a tree, which plain SQL cannot. CMS site maps, category trees and org charts are all trees stored as rows that point at a parent_id. The recursive CTE has two parts joined by UNION ALL: an anchor (the starting rows) and a step that finds the children of the rows found so far. Postgres repeats the step until it finds no new rows.

```sql
-- Concept 1: a recursive CTE builds the full URL path of every page.
WITH RECURSIVE tree AS (
    -- Anchor: the starting rows (pages with no parent, here only home).
    SELECT id, slug, '/' || slug AS path, 0 AS depth
    FROM lesson_pages
    WHERE parent_id IS NULL
    UNION ALL
    -- Step: children of the rows found so far. || joins text,
    -- like + on strings in JS.
    SELECT p.id, p.slug, t.path || '/' || p.slug, t.depth + 1
    FROM lesson_pages p
    JOIN tree t ON p.parent_id = t.id
)
SELECT id, depth, path FROM tree ORDER BY path;
-- ->  id | depth |              path
-- ->   1 |     0 | /home
-- ->   2 |     1 | /home/men
-- ->   3 |     1 | /home/women
-- ->   4 |     2 | /home/women/ethnic-wear
-- ->   5 |     3 | /home/women/ethnic-wear/sarees
-- REMEMBER: anchor UNION ALL step; the step joins the table to the CTE.
-- GOTCHA: if the data has a loop (page 4's parent is 5, 5's is 4), this
-- runs forever. Add WHERE t.depth < 20 in the step as a safety limit.
```

### Real-world FDE example

A retail customer migrating from Drupal to Contentstack needed to rebuild 6,000 old URLs as redirects. Drupal stored the menu as rows with parent ids. One recursive CTE produced every full path (/women/ethnic-wear/sarees) next to the new entry uid, and the redirect map was a single query exported to CSV. The same CTE later checked the migration: any old path without a new uid was a missing page.

### Common mistakes

- Expecting a CTE to exist in the next statement. It belongs to one statement only; use a view for reuse.

- Forgetting the comma between CTEs, or putting one after the last one before the final SELECT.

- Writing UNION instead of UNION ALL in a recursive CTE. UNION removes duplicates on every step, which is slower and can hide real duplicates in your data.

- Naming CTEs t1, t2, t3. The whole point is readable names: totals, latest, ranked.

- Building a ten-step CTE when a single GROUP BY would do. Use steps when they make the query clearer, not by default.

## Concept 2: Window functions, calculations across rows without losing them

### Why it matters for an FDE

GROUP BY answers "what is the total per entry?" but throws away the detail rows. Many business questions need both at once: each day's views and its share of the entry's total, each entry and its rank, the top 3 articles per locale. Before window functions, people solved these with self-joins that were slow and hard to read.

Window functions are one of the most tested SQL topics in data-heavy interviews, and on a customer site they are how you build leaderboards, "top N per group" lists and trend reports. Week 4's FastAPI endpoints and Phase 2's RAG evaluation reports both use them.

### How it works

A window function is a normal aggregate or ranking function followed by OVER (...). The OVER part describes the window: the set of rows this row can see. PARTITION BY splits the rows into groups (like GROUP BY, but nothing is collapsed), and ORDER BY sets the order inside each group. Every input row stays in the output and gets one extra value.

In JavaScript terms, GROUP BY is a reduce() that returns one value per group. A window function is a map() where the callback can also look at the whole group the item belongs to.

```sql
-- Concept 2: GROUP BY collapses rows; a window function keeps every row.
-- 1. GROUP BY: one row per entry, the daily detail is gone.
SELECT uid, sum(views) AS total
FROM lesson_views
WHERE uid IN ('blt102', 'blt103')
GROUP BY uid
ORDER BY uid;
-- ->   uid   | total
-- ->  blt102 |   222
-- ->  blt103 |    95
-- 2. Window: every daily row stays, and also sees its entry's total.
SELECT uid, day, views,
       sum(views) OVER (PARTITION BY uid) AS entry_total,
       round(100.0 * views / sum(views) OVER (PARTITION BY uid), 1)
           AS pct_of_entry
FROM lesson_views
WHERE uid IN ('blt102', 'blt103')
ORDER BY uid, day;
-- ->   uid   |    day     | views | entry_total | pct_of_entry
-- ->  blt102 | 2026-11-01 |    40 |         222 |         18.0
-- ->  blt102 | 2026-11-02 |    45 |         222 |         20.3
-- ->  blt102 | 2026-11-03 |    38 |         222 |         17.1
-- ->  blt102 | 2026-11-04 |    52 |         222 |         23.4
-- ->  blt102 | 2026-11-05 |    47 |         222 |         21.2
-- ->  blt103 | 2026-11-03 |    10 |          95 |         10.5
-- ->  blt103 | 2026-11-04 |    25 |          95 |         26.3
-- ->  blt103 | 2026-11-05 |    60 |          95 |         63.2
-- 8 rows in, 8 rows out. 100.0 (not 100) makes the division decimal.
```

Three ranking functions number the rows in the window's order. They differ only in how they treat ties:

| Function | Ties get | After a tie of two at 2 | Use it for |
|---|---|---|---|
| row_number() | different numbers (order among ties is not fixed) | 2, 3, 4 | exactly N rows per group, latest version |
| rank() | the same number, then a gap | 2, 2, 4 | leaderboards ("joint second") |
| dense_rank() | the same number, no gap | 2, 2, 3 | "top 3 distinct scores" |

Here are all three on the lesson data, where two days tie at 310 views:

```sql
-- Concept 2: three ranking functions on the same order, with a tie.
SELECT uid, day, views,
       row_number() OVER (ORDER BY views DESC) AS row_num,
       rank()       OVER (ORDER BY views DESC) AS rnk,
       dense_rank() OVER (ORDER BY views DESC) AS dense
FROM lesson_views
ORDER BY views DESC
LIMIT 5;
-- ->   uid   |    day     | views | row_num | rnk | dense
-- ->  blt101 | 2026-11-02 |   340 |       1 |   1 |     1
-- ->  blt104 | 2026-11-01 |   310 |       2 |   2 |     2
-- ->  blt101 | 2026-11-03 |   310 |       3 |   2 |     2
-- ->  blt101 | 2026-11-01 |   120 |       4 |   4 |     3
-- ->  blt101 | 2026-11-04 |    90 |       5 |   5 |     4
-- GOTCHA: the two 310 rows may swap row_num 2 and 3 on your machine.
-- row_number() picks any order among ties unless you add a tiebreaker:
-- OVER (ORDER BY views DESC, uid) is the same on every run.
```

Add PARTITION BY and the numbering restarts in every group. That gives you "top N per group", one of the most common report requests. One rule catches everyone the first time: window functions are calculated after WHERE, so you cannot filter on them in the same SELECT. Compute them in a CTE, then filter outside.

```sql
-- Concept 2: top 2 days per entry. PARTITION BY restarts the numbering.
WITH ranked AS (
    SELECT uid, day, views,
           row_number() OVER (PARTITION BY uid
                              ORDER BY views DESC, day) AS rn
    FROM lesson_views
)
SELECT uid, day, views, rn
FROM ranked
WHERE rn <= 2          -- filter OUTSIDE the CTE that computed rn
ORDER BY uid, rn;
-- ->   uid   |    day     | views | rn
-- ->  blt101 | 2026-11-02 |   340 |  1
-- ->  blt101 | 2026-11-03 |   310 |  2
-- ->  blt102 | 2026-11-04 |    52 |  1
-- ->  blt102 | 2026-11-05 |    47 |  2
-- ->  blt103 | 2026-11-05 |    60 |  1
-- ->  blt103 | 2026-11-04 |    25 |  2
-- ->  blt104 | 2026-11-01 |   310 |  1
-- ->  blt104 | 2026-11-05 |    22 |  2
-- The same filter written directly in WHERE fails:
-- -> ERROR:  window functions are not allowed in WHERE
-- REMEMBER: compute the window in a CTE, filter on it outside.
```

### Real-world FDE example

A publisher wanted a homepage module showing "the 3 most-read articles in each section this week", across 14 sections. The first version made 14 queries from the app, one per section, and the page took 2 seconds. One query with row_number() OVER (PARTITION BY section ORDER BY reads DESC) and WHERE rn <= 3 returned all 42 rows at once in 30 milliseconds, and adding a fifteenth section needed no code change.

### Common mistakes

- Putting a window function in WHERE or HAVING. Wrap it in a CTE and filter outside.

- Using row_number() for a leaderboard where ties should share a place. That is rank().

- No tiebreaker in ORDER BY, so "top 2" returns different rows on different runs, and tests fail at random.

- Forgetting PARTITION BY, so the ranking runs over the whole table instead of restarting per group.

- Integer division: 100 * views / total gives whole numbers only. Use 100.0.

## Concept 3: Latest per group, LAG and running totals

### Why it matters for an FDE

Every CMS keeps versions. Contentstack, Drupal and most headless systems store each save, and almost every report starts with "the latest version of each entry". The same shape shows up everywhere else: the latest status of each support ticket, the most recent price of each product, the last login of each user. Getting it right, fast and readable is a daily task.

The other half of analytics is change over time: "up 12% on last week", "running total for the month", "7-day average". Customers judge a dashboard on these numbers, so they must be right on the first day.

### How it works

For the latest row per group, number the rows inside each group newest first and keep number 1. Postgres also has a shortcut, DISTINCT ON, which keeps the first row of each group in the order you give. It is shorter, but it only exists in Postgres; the row_number() version runs on every major database.

```sql
-- Concept 3: the latest version of every entry, two ways.
-- 1. Standard SQL: number versions newest first, keep number 1.
WITH numbered AS (
    SELECT uid, version, title, status,
           row_number() OVER (PARTITION BY uid
                              ORDER BY version DESC) AS rn
    FROM lesson_versions
)
SELECT uid, version, title, status
FROM numbered
WHERE rn = 1
ORDER BY uid;
-- ->   uid   | version |        title         |  status
-- ->  blt101 |       3 | Diwali mega sale     | published
-- ->  blt102 |       1 | Returns policy       | published
-- ->  blt103 |       2 | Winter lookbook 2026 | draft
-- ->  blt104 |       1 | Store locator        | published
-- 2. Postgres only: DISTINCT ON keeps the FIRST row of each uid group,
-- in the ORDER BY order. The ORDER BY must start with the same column.
SELECT DISTINCT ON (uid) uid, version, title, status
FROM lesson_versions
ORDER BY uid, version DESC;
-- -> (the same 4 rows)
-- REMEMBER: latest per group = row_number() ... ORDER BY x DESC, rn = 1.
```

For change over time, lag(col) reads the value from the previous row in the window, and lead(col) the next one. It is like reading arr[i - 1] inside arr.map((x, i, arr) => ...). For running totals and moving averages you add a frame: ROWS BETWEEN ... AND ... says exactly which neighbouring rows the function may see.

```sql
-- Concept 3: day-over-day change, running total and 3-day average.
SELECT uid, day, views,
       lag(views) OVER w                      AS prev_day,
       views - lag(views) OVER w              AS change,
       sum(views) OVER (w ROWS BETWEEN UNBOUNDED PRECEDING
                          AND CURRENT ROW)    AS running_total,
       round(avg(views) OVER (w ROWS BETWEEN 2 PRECEDING
                                AND CURRENT ROW), 1) AS avg_3day
FROM lesson_views
WHERE uid = 'blt101'
WINDOW w AS (PARTITION BY uid ORDER BY day)   -- name the window once
ORDER BY day;
-- -> day        | views | prev_day | change | running_total | avg_3day
-- -> 2026-11-01 |   120 |          |        |           120 |    120.0
-- -> 2026-11-02 |   340 |      120 |    220 |           460 |    230.0
-- -> 2026-11-03 |   310 |      340 |    -30 |           770 |    256.7
-- -> 2026-11-04 |    90 |      310 |   -220 |           860 |    246.7
-- -> 2026-11-05 |    75 |       90 |    -15 |           935 |    158.3
-- (uid column dropped here to fit the page; it is blt101 on every row)
-- The first day has no previous row, so lag() is NULL and so is change.
-- avg_3day on day 1 and 2 averages the 1 or 2 rows that exist.
```

When you write ORDER BY inside OVER and no frame, Postgres uses a default frame: from the first row up to the current row, counting rows with the same ORDER BY value as one block. That default is right for running totals and wrong for last_value(), which is the classic surprise:

```sql
-- Concept 3: the default frame trap with last_value().
SELECT day, views,
       last_value(views) OVER (ORDER BY day)  AS last_default,
       last_value(views) OVER (ORDER BY day
           ROWS BETWEEN UNBOUNDED PRECEDING
                AND UNBOUNDED FOLLOWING)      AS last_whole
FROM lesson_views
WHERE uid = 'blt103'
ORDER BY day;
-- ->     day     | views | last_default | last_whole
-- ->  2026-11-03 |    10 |           10 |         60
-- ->  2026-11-04 |    25 |           25 |         60
-- ->  2026-11-05 |    60 |           60 |         60
-- The default frame ends at the CURRENT row, so the "last" value it sees
-- is the row itself. GOTCHA: for "the final value", give the whole frame,
-- or use first_value() with ORDER BY day DESC.
```

### Real-world FDE example

A customer's weekly email said "traffic up 340%" for a page that went from 5 views to 22. The report compared each week with lag() correctly, but nobody had put a floor on tiny numbers. The fix had two parts: show the absolute change next to the percentage, and only flag pages with at least 500 views in the earlier week. Window functions give you the numbers; deciding which ones a person should see is the FDE's job.

### Common mistakes

- Using max(version) in a subquery per row to find the latest version. It works, but it is slower and harder to extend than row_number() ("latest published version" becomes a rewrite instead of one WHERE).

- DISTINCT ON with an ORDER BY that does not start with the DISTINCT ON columns. Postgres refuses the query.

- Dividing by lag() without thinking about NULL (the first row) and 0 (a week with no views). Both need a decision: NULL is fine, division by zero is an error; NULLIF(prev, 0) avoids it.

- Gaps in the dates. lag() reads the previous ROW, not the previous DAY. If a day is missing, "yesterday" is really two days ago. Fill gaps with generate_series when it matters.

- Trusting the default frame for last_value() or moving averages. Write the frame out.

## Concept 4: JSONB, searching inside flexible CMS fields

### Why it matters for an FDE

Headless CMS content is JSON. A Contentstack entry has fixed fields (uid, title, locale) and a lot of flexible ones: SEO groups, tag lists, modular blocks that differ per content type. Turning every possible field into its own column means a schema change every time an editor adds a field. Storing the whole entry as text means you cannot search it.

Postgres JSONB sits in between: a column that holds JSON in a parsed, binary form you can query and index. Phase 2 uses it directly: a RAG app stores each chunk's metadata (source, section, permissions) as JSONB and filters on it before the vector search. Knowing when to use JSONB, and when not to, is a common system design question.

### How it works

Four operators cover most needs. -> returns a JSON value (still jsonb), ->> returns it as text, #>> follows a path, and @> asks "does this JSON contain that JSON?". The arrows work like entry.seo.meta_title in JavaScript, with one difference: a missing key gives NULL instead of throwing.

```sql
-- Concept 4: reading values out of a JSONB column.
SELECT uid,
       fields -> 'seo' ->> 'meta_title' AS meta_title,  -- text
       fields #>> '{seo,noindex}'       AS noindex,     -- path, as text
       (fields ->> 'discount')::int     AS discount     -- text to integer
FROM lesson_entries
ORDER BY uid;
-- ->   uid   | meta_title  | noindex | discount
-- ->  blt101 | Diwali Sale | false   |       40
-- ->  blt102 | Returns     | true    |
-- ->  blt103 | Winter      |         |       15
-- ->  blt104 |             |         |
-- Missing keys give NULL, never an error (like ?. in JavaScript).
SELECT fields -> 'seo' AS seo FROM lesson_entries WHERE uid = 'blt101';
-- ->  {"noindex": false, "meta_title": "Diwali Sale"}
-- The keys came back in a different order than they were inserted:
-- JSONB stores parsed data, not your original text.
```

Filtering uses the same operators, plus ? which tests whether a key (or an array element that is a string) exists. Containment with @> is the one to learn first, because it is the form an index can speed up.

```sql
-- Concept 4: filtering on JSONB.
-- 1. Entries whose tags array includes "festive".
SELECT uid FROM lesson_entries
WHERE fields -> 'tags' ? 'festive'
ORDER BY uid;
-- -> blt101, blt103
-- 2. The same with containment: "fields contains this JSON".
SELECT uid FROM lesson_entries
WHERE fields @> '{"tags": ["festive"]}'
ORDER BY uid;
-- -> blt101, blt103
-- 3. Pages hidden from search engines (nested containment).
SELECT uid FROM lesson_entries
WHERE fields @> '{"seo": {"noindex": true}}';
-- -> blt102
-- 4. Entries with no seo block at all: ? on the top level.
SELECT uid FROM lesson_entries WHERE NOT fields ? 'seo';
-- -> blt104
-- 5. Numbers need a cast, because ->> always returns text.
SELECT uid FROM lesson_entries
WHERE (fields ->> 'discount')::int >= 20;
-- -> blt101
```

Arrays inside JSON often need to become rows, for example to count tags. jsonb_array_elements_text() does that: it is a function that returns a set of rows, so you put it in the FROM list next to the table, and each entry row is paired with each of its tags.

```sql
-- Concept 4: one row per tag, then count like any other column.
SELECT tag, count(*) AS entries
FROM lesson_entries,
     jsonb_array_elements_text(fields -> 'tags') AS tag
GROUP BY tag
ORDER BY entries DESC, tag;
-- ->    tag   | entries
-- ->  festive |       2
-- ->  diwali  |       1
-- ->  help    |       1
-- ->  sale    |       1
-- ->  winter  |       1
-- blt104 has an empty tags array, so it produces no rows at all.
```

On a big table, @> without an index reads every row, the Seq Scan you met on Day 19. A GIN index (generalised inverted index) works like the index at the back of a book for every key and value inside the JSON, the same idea as a search index such as Algolia over your entries.

```sql
-- Concept 4: a GIN index makes @> fast. Builds 100,000 rows (a few sec).
DROP TABLE IF EXISTS lesson_big;
CREATE TABLE lesson_big AS
SELECT 'blt' || g AS uid,
       jsonb_build_object(
           'tags', CASE WHEN g % 100 = 0 THEN '["festive"]'::jsonb
                        ELSE '["evergreen"]'::jsonb END,
           'n', g) AS fields
FROM generate_series(1, 100000) AS g;      -- 1 row in 100 is festive
ANALYZE lesson_big;
SET max_parallel_workers_per_gather = 0;   -- one process: simpler plans
EXPLAIN ANALYZE SELECT count(*) FROM lesson_big
WHERE fields @> '{"tags": ["festive"]}';
-- ->  Seq Scan on lesson_big (actual time=0.037..32.372 rows=1000)
-- ->    Rows Removed by Filter: 99000
-- ->  Execution Time: 32.626 ms
CREATE INDEX lesson_big_fields_gin ON lesson_big USING gin (fields);
EXPLAIN ANALYZE SELECT count(*) FROM lesson_big
WHERE fields @> '{"tags": ["festive"]}';
-- ->  Bitmap Heap Scan on lesson_big (actual time=1.205..2.247 rows=1000)
-- ->    ->  Bitmap Index Scan on lesson_big_fields_gin
-- ->  Execution Time: 2.352 ms
-- About 14 times faster here. Plan lines shortened; your times differ.
-- GOTCHA: this index helps @>, ? and similar operators. It does NOT help
-- (fields ->> 'discount')::int >= 20; that needs an expression index.
```

When to use a column and when to use JSONB:

| Question | Regular column | JSONB |
|---|---|---|
| Every row has it, and you filter, join or sort on it | yes (uid, locale, status) | no |
| It needs a rule (NOT NULL, CHECK, FOREIGN KEY) | yes | rules are hard to write |
| Shape differs per content type or changes often | a schema change each time | yes (SEO groups, modular blocks) |
| You store the source system's payload as it came | no | yes (keep the raw entry for audits) |

### Real-world FDE example

A beauty brand's Contentstack had 38 content types with different modular blocks. The FDE team synced each entry into Postgres with five real columns (uid, content_type, locale, status, updated_at) and the rest of the entry in a JSONB column with a GIN index. Editors' questions like "which product pages have no SEO description?" became one query with ? and @>, without a migration for each new field. When one JSON field became central to every report (the product SKU), it was promoted to a real column with an index.

### Common mistakes

- Comparing ->> text as if it were a number: '40' > '5' is false, because text compares character by character. Cast first with ::int or ::numeric.

- Using -> where you meant ->>. fields -> 'title' = 'Diwali' fails with "invalid input syntax for type json": the left side is jsonb, so Postgres tries to read 'Diwali' as JSON.

- Putting everything in JSONB, including columns you join and filter on in every query. Those belong in real columns with constraints.

- Expecting a GIN index to speed up every JSON expression. It serves containment and key tests; casts and comparisons need an expression index.

- Using json instead of jsonb. json keeps the raw text and cannot be indexed the same way; jsonb is the default choice.

## Concept 5: pandas read_sql, from a query to a report

### Why it matters for an FDE

SQL answers the question; people still want a spreadsheet, a chart or a CSV attached to an email. pandas is the Python library for tables in memory, and read_sql() turns a query result into a DataFrame in one line. From there, grouping, pivoting and writing to CSV or Excel are each one line too.

On a customer site, this is the quickest way to answer "can you send us the numbers?" without building a dashboard, and in Phase 2 it is how you analyse eval results (scores per prompt version, failures per category). Week 9's data skills build on this.

### How it works

A DataFrame is a table with named columns, each with one data type. Think of it as an array of row objects in JavaScript, stored column by column, with spreadsheet operations built in. pandas reads from a database through SQLAlchemy, a library that wraps database drivers in one common interface. You give it a URL; the +psycopg part tells SQLAlchemy to use the psycopg 3 driver you used on Day 18.

In [ ]:
# Concept 5: run SQL from pandas and get a DataFrame back.
import os

import pandas as pd                        # "pd" is the usual short name
from sqlalchemy import create_engine

DB_URL = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")
# SQLAlchemy picks the driver from the URL: "+psycopg" means psycopg 3.
engine = create_engine(DB_URL.replace("postgresql://",
                                      "postgresql+psycopg://", 1))
# An engine is a connection factory: it connects when a query runs.

df = pd.read_sql("SELECT uid, day, views FROM lesson_views", engine)
print(df.shape)              # (rows, columns), like a tuple from Day 3
# -> (18, 3)
print(df.dtypes)             # one data type per column
# -> uid         str
# -> day      object
# -> views     int64
print(df.head(3))            # first 3 rows; the left column is the index
# ->       uid         day  views
# -> 0  blt101  2026-11-01    120
# -> 1  blt101  2026-11-02    340
# -> 2  blt101  2026-11-03    310
# GOTCHA: day is "object" (Python date objects), not a real date column,
# so .dt date methods will not work on it yet. The next example fixes it.

pandas can also take a raw psycopg connection, but it warns: "pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested." Use an engine and the warning never appears.

Two habits from Day 18 carry over. Values go in as parameters, never pasted into the SQL text: with SQLAlchemy you write :name in a text() query and pass params. And you should tell pandas which columns are dates with parse_dates, so you can group by week or format them.

In [ ]:
# Concept 5: parameters, real dates, groupby and a pivot table.
import os

import pandas as pd
from sqlalchemy import create_engine, text

DB_URL = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")
engine = create_engine(DB_URL.replace("postgresql://",
                                      "postgresql+psycopg://", 1))

# text() + :since: the value travels separately from the SQL (Day 18).
SQL = text("""
    SELECT v.uid, e.content_type, v.day, v.views
    FROM lesson_views v
    JOIN lesson_entries e USING (uid)
    WHERE v.day >= :since
""")
df = pd.read_sql(SQL, engine, params={"since": "2026-11-02"},
                 parse_dates=["day"])          # day becomes datetime64
print(df.dtypes["day"])
# -> datetime64[s]

# groupby + agg: GROUP BY in memory. Each name=("col", "func") pair
# becomes one output column, like SUM(...) AS total in SQL.
by_type = (df.groupby("content_type")["views"]
             .agg(total="sum", best_day="max")
             .sort_values("total", ascending=False))
print(by_type)
# ->               total  best_day
# -> content_type
# -> campaign        910       340
# -> page            257        52

# pivot_table: one row per entry, one column per day, like a spreadsheet.
grid = df.pivot_table(index="uid", columns=df["day"].dt.strftime("%d %b"),
                      values="views", aggfunc="sum", fill_value=0)
print(grid)
# -> day     02 Nov  03 Nov  04 Nov  05 Nov
# -> uid
# -> blt101     340     310      90      75
# -> blt102      45      38      52      47
# -> blt103       0      10      25      60
# -> blt104      15      20      18      22
grid.to_csv("views_grid.csv")      # a file the editors open in Excel
# GOTCHA: blt103's 0 on 02 Nov is fill_value. The database has no row
# that day. Say so in the report if "no data" and "0 views" differ.

Where should the work happen? A short rule that holds up in practice:

| Do it in SQL | Do it in pandas |
|---|---|
| filtering rows (WHERE) before they leave the database | reshaping for people: pivots, wide grids |
| joins, latest-per-group, window functions | quick exploration and one-off questions |
| aggregating millions of rows down to hundreds | writing CSV, Excel and charts |
| anything a FastAPI endpoint or another tool will reuse | combining with data that is not in the database (a CSV from the customer) |

### Real-world FDE example

On a support-bot pilot, the customer's operations lead asked every Friday for "tickets the bot solved, by category and week". The first version pulled 2 million ticket rows into pandas and grouped them there; it used 6 GB of memory and took four minutes. The rewrite grouped by category and week in SQL, returned 300 rows, and used pandas only for the pivot and the Excel file. Same report, under two seconds, and it ran in the customer's small CI container.

### Common mistakes

- Passing a raw psycopg connection and ignoring the warning. Use a SQLAlchemy engine.

- Building the SQL with an f-string because "it is only a report". Reports get parameters from users too; use :name and params.

- Pulling whole tables into pandas and filtering there. Filter and aggregate in SQL first; use chunksize= in read_sql if you really must stream a big table.

- Forgetting parse_dates, then fighting with strings or Python date objects when grouping by week.

- Reporting fill_value=0 as "0 views" when the truth is "no data for that day".

## Check your understanding (predict first, then run)

Write down what each numbered query returns, then run the file in psql inside day20/ with fresh lesson tables. Answers are on the last page.

```sql
-- Predict, then run. Re-run setup.sql before you start.
-- 1: three ranking functions on 50, 50, 40. Fill in rn, rk and dr.
SELECT v, row_number() OVER (ORDER BY v DESC) AS rn,
       rank()       OVER (ORDER BY v DESC) AS rk,
       dense_rank() OVER (ORDER BY v DESC) AS dr
FROM (VALUES (50), (50), (40)) AS t(v);
-- 2: a running total ordered by day only. Two entries share each day.
-- What is in the running column on each of the 4 rows?
SELECT uid, day, views,
       sum(views) OVER (ORDER BY day) AS running
FROM lesson_views
WHERE uid IN ('blt102', 'blt103') AND day >= '2026-11-04'
ORDER BY day, uid;
-- 3: discount is 40 for blt101 and 15 for blt103. Which rows come back?
SELECT uid FROM lesson_entries
WHERE fields ->> 'discount' > '5'
ORDER BY uid;
-- 4: two entries have "festive" in their tags. What does this count?
SELECT count(*) FROM lesson_entries WHERE fields ? 'festive';
```

## Hands-on task (about 2 hours): day20/analytics.sql and report.py

Acme Retail's content lead sends the same three questions every Monday: what is live and what is waiting in draft, what is getting views this festive season, and "can I have it in Excel?". Your job: answer the questions in SQL a teammate can read, then wrap the weekly numbers in a small pandas report. Week 4 puts both behind FastAPI endpoints, so the queries must be correct, ordered and reusable.

```bash
# day20/ layout for today's task. Everything except the SELECT 'TODO'
# lines in analytics.sql, the TODO functions in report.py and dsa.py
# is given.
day20/
    setup.sql            # lesson tables for the concept examples
    task_data.sql        # Acme's cms_versions (20 saves of 12 entries)
                         # and cms_daily_views (4 weeks, 280 rows)
    analytics.sql        # YOUR 8 queries, A1 to A8
    test_analytics.py    # 8 given tests: runs each query, checks rows
    report.py            # YOUR weekly report: 5 TODOs
    test_report.py       # 14 given tests (9 need no database)
    requirements.txt     # psycopg 3, SQLAlchemy 2, pandas, pytest
    dsa.py               # today's two DSA problems
    day20_notes.ipynb    # today's examples, already run
```

Step 1 (10 min): run psql "$DATABASE_URL" -f task_data.sql, then look at the data: SELECT * FROM cms_versions ORDER BY uid, version; and \d cms_daily_views. Write down which entries have more than one version, which were never published, and which have no views.

Step 2 (45 min): write A1 to A5 in analytics.sql. Try each in psql first, then paste it in, and run python -m pytest -q test_analytics.py after each one. A failing test tells you whether the columns, the row count or the values differ.

Step 3 (25 min): write A6 (week over week with lag), A7 and A8 (JSONB). All 8 tests should pass.

Step 4 (35 min): write report.py: DAILY_SQL and make_engine first, then summarise_by_type and weekly_grid (their tests need no database), then fetch_daily and main.

Step 5 (5 min): do the four manual runs below, open weekly_views.csv in Excel or Numbers, and run the whole suite.

```bash
# Steps 2 to 5: what correct runs look like.
$ python -m pytest -q test_analytics.py
........                                                     [100%]
8 passed in 0.13s
$ python report.py
rows: 280, entries: 10, since: 2026-10-05
              entries  total_views  avg_per_entry
content_type
campaign            4        48541        12135.2
article             2        13664         6832.0
page                4        13082         3270.5
wrote summary_by_type.csv and weekly_views.csv
$ python report.py 2026-10-26
rows: 70, entries: 10, since: 2026-10-26
              entries  total_views  avg_per_entry
content_type
campaign            4        16722         4180.5
article             2         4716         2358.0
page                4         3786          946.5
wrote summary_by_type.csv and weekly_views.csv
$ python report.py 2027-01-01
no views since 2027-01-01
$ DATABASE_URL=postgresql://localhost:5999/fde python report.py
error: cannot reach the database; is Postgres running?
$ echo $?
1
$ python -m pytest -q
22 passed in 0.66s
```

Acceptance criteria:

- python -m pytest -q inside day20/ shows 22 passed (8 in test_analytics.py, 14 in test_report.py).

- The four manual runs print exactly the lines above and exit with those codes. Errors go to stderr, and no run prints a traceback.

- Every query that needs the latest version gets it from a CTE named latest, built with row_number() or DISTINCT ON. No query uses a correlated max(version) subquery.

- Every query in analytics.sql ends with an ORDER BY, and every CTE and output column has a clear name.

- The since date reaches SQL through :since and params; no f-string or + builds SQL text anywhere.

- python -c "import report" inside day20/ prints nothing.

- Committed and pushed with a meaningful message; .venv/ and the generated CSV files are not committed.

Hints (no solutions, try first):

- A2 is A1 with one extra condition. Where does it go: before the numbering (in the CTE) or after it (rn = 1)? Try both and compare blt003.

- A3 joins two CTEs. If you get blt010 or blt012 in the result, your join keeps entries that were never published.

- A4: count the rows of your totals CTE before ranking. If it has 10 rows instead of 12, the LEFT JOIN or the COALESCE is missing.

- A6: build the weekly CTE first and check it has 8 rows for the two entries. Only then add lag().

- A7: run your noindex condition on its own over all latest rows. blt009 (noindex true) must be out; blt005 (no noindex key at all) must be in.

- weekly_grid: print df["day"].dt.to_period("W-SUN").head() to see what a weekly period looks like before you convert it.

- If a report.py test fails with a KeyError on a column name, print out.columns and compare with the names in the test.

## DSA practice (30 min)

Both problems are in day20/dsa.py with signatures, examples and hints. Run python dsa.py; every line should print True.

- Latest by key (easy). latest_by_key(records) takes (uid, version, title) tuples in any order and returns {uid: title of the highest version}: today's "latest version per entry" without a database. Hint: one pass with a dict that stores the best (version, title) seen so far for each uid.

- Maximum window sum (medium). max_window_sum(nums, k) returns the largest total of k consecutive days, in O(n). This is the moving frame from Concept 3. Hint: when the window slides one step, one number enters and one leaves; you never need to re-add the whole window.

## Quiz (3 quick questions)

- A teammate finds the latest version per entry with WHERE version = (SELECT max(version) FROM cms_versions v2 WHERE v2.uid = v.uid). The customer now wants "latest published version, and the version before it". Why is row_number() the better starting point?

- Acme wants to store each entry's SEO block. One proposal: three columns (meta_title, meta_description, noindex). The other: one JSONB column for all SEO fields. Which would you choose for an FDE engagement where content types keep changing, and what would make you move a field out of JSONB?

- The content lead says your report shows "0 views" for a new page on a day before it was published. What happened in the code, and what would you change?

## What you achieved today

You can now write SQL in named steps, rank and compare rows without losing them, pick the latest version of anything, measure change over time, search inside JSON documents, and turn the result into a file people can open. That covers most of the reporting questions a content or operations team will ask you on a customer site, and it is the query layer Week 4's API will expose.

Tomorrow (Day 21): weekly review of Week 3 with scores per area, a mock interview prompt, the catch-up lane closed out, and the Week 4 plan (FastAPI).

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 20 review": your recap answers, your four predictions next to the real outputs, analytics.sql, report.py, the pytest summary line, the terminal output of the four manual runs, your quiz answers, dsa.py, and the link to your latest commit. Say how far the Day 19 catch-up got.

## Recap answers

*Your answers:*
